# Clase 001 — Instalación de Python 3.12+ y entornos virtuales

**Parte 0 — Prerrequisitos** · Fuente: VanderPlas, *Python Data Science Handbook*, prefacio.

> 🎯 **Objetivo de la clase:** dejar tu máquina lista para data science. Python 3.12+, al menos dos gestores de entornos virtuales funcionando, y la disciplina de **nunca tocar el Python del sistema**.

> ⏱️ **Duración:** ~90 min (45 lectura + 45 práctica)

Antes de empezar, lee el [`README.md`](README.md) de esta clase: ahí están los resultados de aprendizaje, los temas y los ejercicios formales.

## 🗺️ Agenda del notebook

1. **Diagnóstico** — ¿qué Python tengo ahora mismo? ¿desde dónde se ejecuta?
2. **Por qué un venv** — el problema concreto que resuelven los entornos virtuales.
3. **`venv` (stdlib)** — crear, activar, instalar, congelar, destruir.
4. **`uv` (Astral)** — alternativa moderna, 10–100× más rápida.
5. **`conda` / `mamba`** — cuándo lo necesitas (y cuándo no).
6. **El bug clásico** — "pip install funcionó pero `import` falla".
7. **Ejercicios** — 5 consignas prácticas.
8. **Checklist + homework**.

## ⚙️ Setup

Imports mínimos: sólo stdlib. Esta clase no requiere paquetes externos — el punto es justamente *aprender a instalarlos*.

In [ ]:
import sys
import platform
import shutil
from pathlib import Path

print('python      :', sys.version.split()[0])
print('executable  :', sys.executable)
print('platform    :', platform.system(), platform.release())
print('cwd         :', Path.cwd())

## 1️⃣ Diagnóstico — ¿qué Python tengo?

Antes de instalar nada nuevo, **mira qué tienes**. Tres preguntas:

1. ¿Qué versión es? → `python -V` o `sys.version`
2. ¿Desde dónde se ejecuta? → `where python` (Windows) / `which python` (Unix) o `sys.executable`
3. ¿Qué `pip` está asociado a *este* Python? → `python -m pip -V` (NO sólo `pip -V`, que puede ser otro)

**Regla de oro:** siempre usa `python -m pip ...` en lugar de `pip ...` directo. Te garantiza que instalas en el mismo Python que estás ejecutando.

In [ ]:
# Diagnóstico inline desde el notebook
print('Python ejecutando este notebook:')
print('  versión   :', sys.version.split()[0])
print('  ruta      :', sys.executable)
print()
print('¿Es un venv?', '.venv' in sys.executable or 'envs' in sys.executable)
print('Prefix      :', sys.prefix)
print('Base prefix :', sys.base_prefix)
print()
print('Si sys.prefix != sys.base_prefix, estás dentro de un entorno virtual ✅')

## 2️⃣ ¿Por qué un entorno virtual? El problema concreto

Imagina dos proyectos en tu máquina:

- **Proyecto A** (legacy): necesita `pandas==1.5.3` porque usa `df.append()` (eliminado en 2.0).
- **Proyecto B** (nuevo): necesita `pandas==2.2.3` para `pd.concat` con `Index.union(sort=False)`.

Sin venv, instalar uno **rompe** el otro. Con venv, cada proyecto tiene su propio Python aislado, con sus propias dependencias, y no se pisan.

**En data science esto es crítico** porque:
- Reproducibilidad: tu notebook debe correr igual en 6 meses.
- Experimentos: probar `scikit-learn 1.6` no debe romper el modelo que tienes en producción con `1.4`.
- Onboarding: un compañero clona el repo y reproduce tu entorno exacto en 30 segundos.

## 3️⃣ `venv` — el gestor nativo de Python

Viene incluido con Python desde la versión 3.3 (PEP 405). Es el **default seguro**: no requiere instalar nada extra.

**Ciclo de vida completo de un venv:**

```bash
# 1. Crear (genera una carpeta .venv/ con su propio Python)
python -m venv .venv

# 2. Activar
source .venv/bin/activate         # macOS / Linux
.venv\Scripts\Activate.ps1        # Windows PowerShell
.venv\Scripts\activate.bat        # Windows CMD

# 3. Verificar que el prompt cambió y python apunta al venv
where python                       # Windows
which python                       # Unix

# 4. Instalar paquetes (sólo afectan a este venv)
python -m pip install numpy pandas matplotlib jupyter

# 5. Congelar versiones exactas para reproducir después
python -m pip freeze > requirements.txt

# 6. Salir del venv
deactivate

# 7. Eliminarlo (es sólo una carpeta — bórrala)
rm -rf .venv      # Unix
Remove-Item -Recurse -Force .venv  # PowerShell
```

In [ ]:
# Demostración: el notebook reporta qué venv estamos usando (si alguno)
venv_marker = Path(sys.prefix) / 'pyvenv.cfg'
if venv_marker.exists():
    print('✅ Estás en un venv. Contenido de pyvenv.cfg:')
    print(venv_marker.read_text())
else:
    print('⚠️  Estás en el Python global / del sistema.')
    print('    Considera crear un venv antes de instalar paquetes.')

## 4️⃣ `uv` — el reemplazo moderno (Astral, 2024+)

`uv` hace lo mismo que `pip` + `venv` + `pip-tools`, pero **10–100× más rápido** (escrito en Rust). Drop-in replacement: si sabes pip, ya sabes uv.

**Instalación de uv** (una sola vez por máquina, fuera de cualquier venv):

```bash
# Opción A: instalador oficial (recomendado)
curl -LsSf https://astral.sh/uv/install.sh | sh    # Unix
powershell -c "irm https://astral.sh/uv/install.ps1 | iex"   # Windows

# Opción B: vía pipx
pipx install uv
```

**Ciclo equivalente al de venv, pero con uv:**

```bash
uv venv                              # crea .venv con Python detectado
uv venv --python 3.12                # crea .venv forzando 3.12 (lo descarga si falta)
source .venv/bin/activate            # activación igual que con venv
uv pip install numpy pandas          # ← esto vuela
uv pip freeze > requirements.txt
```

**Bonus:** `uv` también administra versiones de Python (`uv python install 3.12`), reemplazando a `pyenv`.

## 5️⃣ `conda` / `mamba` — cuándo lo necesitas

Conda **no es solo un gestor de paquetes Python** — es un gestor de paquetes binarios multilenguaje (Python + C + CUDA + R + …). Eso lo hace pesado, pero **imprescindible** cuando:

- Necesitas **PyTorch o TensorFlow con GPU** y CUDA preempaquetado (sin pelearte con drivers).
- Trabajas con **geopandas**, **rdkit**, **GDAL**, **pdal** — paquetes con dependencias C/C++ nativas difíciles de compilar con pip.
- Quieres aislar **Python *y* librerías de sistema** (un compilador específico, una versión de OpenSSL).

**Si tu proyecto es puramente Python (numpy/pandas/sklearn/matplotlib), prefiere `venv` o `uv`.** Más rápido, más portable, menos peso.

**Mínimo recomendado: Miniconda** (no Anaconda completo, que pesa 3 GB con cosas que no usarás).

```bash
conda create -n ds-2026 python=3.12   # crea entorno llamado ds-2026
conda activate ds-2026
conda install -c conda-forge numpy pandas matplotlib jupyter
conda env export > environment.yml      # equivalente a requirements.txt
conda deactivate
conda env remove -n ds-2026             # eliminar
```

**`mamba`** es un drop-in replacement de conda mucho más rápido (también escrito en C++). Si usas conda en serio, instálalo: `conda install -n base -c conda-forge mamba`.

## 📊 Comparativa rápida

| Característica | `venv` | `uv` | `conda` |
|---|---|---|---|
| Viene con Python | ✅ stdlib | ❌ instalar | ❌ instalar (Miniconda) |
| Velocidad install | 🐢 baseline | 🚀 10–100× | 🐢🐢 (mamba: 🚀) |
| Maneja versiones de Python | ❌ | ✅ | ✅ |
| Paquetes no-Python (CUDA, GDAL) | ❌ | ❌ | ✅ |
| Formato lockfile | `requirements.txt` | `uv.lock` / `requirements.txt` | `environment.yml` |
| Tamaño en disco | mínimo | mínimo | grande |
| Recomendado para… | proyectos Python puros, simplicidad | lo mismo, pero rápido | GPU, geo, química |

## 6️⃣ El bug clásico: "`pip install` funcionó pero `import` falla"

**Síntoma:** ejecutas `pip install seaborn`, sale `Successfully installed seaborn-0.13.2`. Vas al notebook, escribes `import seaborn` → `ModuleNotFoundError`.

**Causa:** `pip` y `python` apuntan a Pythons distintos. Ejemplo típico:
- `pip` en tu PATH es el del Python global del sistema → instaló ahí.
- Tu notebook corre con el Python de un venv (o de conda) → no lo ve.

**Diagnóstico — siempre estos dos comandos:**

```bash
python -c 'import sys; print(sys.executable)'
pip -V                                # primera línea muestra desde qué Python
```

Si los dos no apuntan a la misma carpeta, ahí está tu bug.

**Solución definitiva:** **siempre** instala con `python -m pip install ...` (en vez de `pip install ...`). Garantiza que pip usa exactamente el Python que invocaste.

In [ ]:
# Diagnóstico programático del bug
pip_path = shutil.which('pip')
python_path = sys.executable

print(f'python ejecutando este notebook : {python_path}')
print(f'pip en el PATH                  : {pip_path}')
print()

# Heurística: ambos deberían vivir bajo el mismo prefix
if pip_path and Path(pip_path).parent.parent == Path(python_path).parent.parent:
    print('✅ pip y python están alineados — instalar con `pip install X` es seguro.')
else:
    print('⚠️  pip y python NO están alineados.')
    print('    Usa SIEMPRE: python -m pip install <paquete>')

## 🧪 Ejercicios

Hazlos en una **terminal real** (no dentro de este notebook). Anota outputs y conclusiones en un archivo aparte.

### Ejercicio 1 — Diagnóstico inicial

Abre una terminal limpia y reporta:
- `python -V`
- `where python` (Windows) o `which python` (Unix)
- Output de un script con `import sys; print(sys.path)`

Guarda los tres outputs como baseline.

### Ejercicio 2 — Tu primer venv

```bash
mkdir lab-001 && cd lab-001
python -m venv .venv
# activa según tu OS
python -m pip install numpy==2.1.0 pandas==2.2.3
python -m pip list
python -c 'import numpy; print(numpy.__version__)'
deactivate
python -c 'import numpy'   # debería fallar si tu Python global no tiene numpy
```

### Ejercicio 3 — Mismo entorno con `uv`

Instala uv, recrea el ejercicio 2 con `uv venv` + `uv pip install`. Compara la velocidad (`time` en Unix, `Measure-Command` en PowerShell).

### Ejercicio 4 — Reproducibilidad

Con el venv del ejercicio 2 activo:
```bash
python -m pip freeze > requirements.txt
deactivate
rm -rf .venv          # o Remove-Item -Recurse
python -m venv .venv
# activa
python -m pip install -r requirements.txt
python -m pip list    # ¿coinciden las versiones?
```

### Ejercicio 5 — Provoca y resuelve el bug clásico

Desde un Jupyter ejecutándose con tu Python global (no el venv), corre en una celda:
```python
!pip install seaborn
import seaborn   # observa qué pasa
import sys; print(sys.executable)   # diagnóstico
```
Explica con tus palabras qué ocurrió y cómo lo arreglarías usando `%pip` o `python -m pip`.

## ✅ Checklist de auto-evaluación

Marca cada ítem antes de pasar a la siguiente clase:

- [ ] Sé qué versión de Python tengo y desde dónde se ejecuta.
- [ ] Creé, activé y destruí al menos un `venv` desde cero.
- [ ] Instalé `uv` y creé un entorno con él.
- [ ] Generé un `requirements.txt` y reproduje el entorno borrándolo y recreándolo.
- [ ] Sé explicar por qué `python -m pip install` es más seguro que `pip install` solo.
- [ ] Sé cuándo elegiría conda en vez de venv/uv.

## 📝 Homework verificable

Entrega un repo (o carpeta zip) con:

1. `README.md` con tu OS, versión de Python y gestor elegido.
2. `.gitignore` que ignore `.venv/` y `__pycache__/`.
3. `requirements.txt` con: `numpy>=2.0`, `pandas>=2.2`, `matplotlib>=3.8`, `jupyter>=1.0`.
4. `verify.py` que imprima `sys.version`, `sys.executable`, `numpy.__version__`, `pandas.__version__`.
5. Output de `python verify.py` pegado al final del `README.md`.

**Criterio de aceptación:** otra persona clona tu repo, hace `python -m venv .venv` → activa → `pip install -r requirements.txt` → `python verify.py` y obtiene un output análogo al tuyo.

Ver `README.md` de esta clase para el detalle completo.

## 🔗 Referencias

- VanderPlas, *Python Data Science Handbook*, **Preface — Installation Considerations**
- [PEP 405 — Python Virtual Environments](https://peps.python.org/pep-0405/)
- [Astral uv — docs oficiales](https://docs.astral.sh/uv/)
- [Miniconda — instalador mínimo](https://docs.conda.io/projects/miniconda/)

---

➡️ **Siguiente clase:** [002 — Jupyter y JupyterLab: kernels, magics, debugging, profiling](../002-jupyter-y-jupyterlab-kernels-magics-debugging-profiling/README.md)